# 📊 **Exploratory Data Analysis — Bitcoin (BTC)**

### 🎯 **Objective**
Explore the Bitcoin **Gold dataset** to understand market behavior and identify patterns associated with **bearish reversals (crashes)**.

### ⚠️ **Target Definition**
A crash is defined as a price drop of more than **10% within the next 7 days**.

- `1` → Crash  
- `0` → No crash  

### 🔬 **EDA Focus**
1.  Distribution of the target (class balance)  
2.  BTC price with crash events  
3.  Drawdown analysis  
4.  Feature distributions  
5.  Correlation between features and target  

### 🚀 **Goal**
Identify signals and patterns that could help predict market reversals.

In [57]:
import random
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### **1. Distribution of the target (class balance)**  

In [58]:
df = pd.read_csv("../data/gold/market/btc_usdt_1d_features.csv", low_memory=False)

DF's infos

In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3134 entries, 0 to 3133
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   open_time               3134 non-null   str    
 1   open                    3134 non-null   float64
 2   high                    3134 non-null   float64
 3   low                     3134 non-null   float64
 4   close                   3134 non-null   float64
 5   volume                  3134 non-null   float64
 6   quote_asset_volume      3134 non-null   float64
 7   number_of_trades        3134 non-null   int64  
 8   taker_buy_base_volume   3134 non-null   float64
 9   taker_buy_quote_volume  3134 non-null   float64
 10  return_1d               3134 non-null   float64
 11  return_7d               3134 non-null   float64
 12  volatility_7d           3134 non-null   float64
 13  volatility_30d          3111 non-null   float64
 14  drawdown                3134 non-null   float64
 15

Focus on target column

In [60]:
print(df["target"].value_counts(normalize=True), "\n\n", df["target"].value_counts())

target
False    0.889598
True     0.110402
Name: proportion, dtype: float64 

 target
False    2788
True      346
Name: count, dtype: int64


**Observation:**  
The dataset is imbalanced, with significantly more non-peak observations than peak events.

**Impact:**  
The model may be biased toward predicting the majority class (no peak), leading to misleading accuracy and poor detection of actual peaks (low recall and/or precision).

**Solution:**
- Use appropriate evaluation metrics (precision, recall, F1-score)  
- Apply class weighting to handle imbalance  
- Use resampling techniques (oversampling or undersampling)  
- Adjust the decision threshold to improve detection of peaks  

### **2. BTC price with crash events**

**Bitcoin Price Over Time**

In [61]:
figure=px.line(df, x="open_time", y="close", title="BTC/USDT Price Over Time")
figure.show()

**Bitcoin Price with Peak Events**

In [62]:
figure=px.line(df,x="open_time", y="close")

peaks = df[df["target"] == 1]

figure.add_scatter(
    x=peaks["open_time"], 
    y=peaks["close"],
    mode="markers",
    marker=dict(color="red", size=6),
    name="Peaks"
    )
figure.update_layout(title="BTC/USDT Price Over Time with Peaks")
figure.show()

**Observations:**
- Peaks are mostly located at local price maxima  
- Peaks often precede significant price declines  
- Strong alignment with major market cycles (bull → top → drop)  
- Some peaks occur during minor fluctuations (noise)  
- Overall, peak detection is consistent with market tops  
- Visual inspection confirms that most detected peaks align with actual market tops  
- The definition captures major turning points but may be slightly sensitive to short-term volatility  

**Price Behavior Around Peaks**

*Analyze the typical price pattern before and after randomly selected peak events, using relative returns centered at the peak (t = 0).*


In [63]:
NB_PEAKS_TO_DISPLAY = 5

peaks_position = random.sample(range(len(peaks)), NB_PEAKS_TO_DISPLAY)
random_peaks = peaks.iloc[peaks_position]

# 2 subplots (1 ligne, 2 colonnes)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Absolute Variation Around Peaks",
        "Relative Returns Around Peaks"
    ]
)

for index, row in random_peaks.iterrows():
    # skip bords
    if index < 7 or index > len(df) - 7:
        continue

    df_slice = df.iloc[index-7:index+7]

    x = list(range(-7, 7))

    # Absolute
    y_abs = df_slice["close"] - row["close"]

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y_abs,
            mode="lines",
            opacity=0.4,
            showlegend=False
        ),
        row=1, col=1
    )

    # Relative
    y_rel = (df_slice["close"] / row["close"]) - 1

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y_rel,
            mode="lines",
            opacity=0.4,
            showlegend=False
        ),
        row=1, col=2
    )

# Layout
fig.update_layout(
    title="Price Behavior Around Peaks (Absolute vs Relative)",
)

fig.update_xaxes(title_text="Days Relative to Peak", row=1, col=1)
fig.update_xaxes(title_text="Days Relative to Peak", row=1, col=2)

fig.update_yaxes(title_text="Price Difference", row=1, col=1)
fig.update_yaxes(title_text="Return vs Peak", row=1, col=2)

fig.show()

**Observations:**
- Prices generally trend upward before labeled peak events  
- A local maximum is often observed around t = 0, but not always exactly aligned  
- In several cases, higher prices occur before t = 0 (e.g. +10%), indicating the true peak happened earlier  
- This shift is due to the target capturing the onset of a crash rather than the exact market top  
- A decline typically follows, confirming the reversal dynamic  
- The magnitude and timing of drops vary across events, reflecting market noise and heterogeneity  
- Overall, the pattern reflects a transition zone: late uptrend → peak region → decline  

### **3.  Drawdown analysis**

Visualize drawdown over time

Identify major correction periods

Compare drawdown with peak events

Analyze depth and duration of declines

Check consistency with market cycles

### **4.  Feature distributions**

Visualize distribution of each feature

Identify outliers

Compare distributions between peaks and non-peaks

Analyze spread and shape of distributions

Detect unusual behavior before peaks

### **5.  Correlation between features and target**

Compute correlation between features and target

Identify features most related to peaks

Analyze direction of relationships (positive/negative)

Detect redundancy between features

Select most relevant features for modeling